# 05d — Theoreme hierarchique a restrictions patient (port fidele)

[< Serie Z3](README.md) | Capstone MealPlanner **Slice C** (*Part of* #4617, *See* #1206)

Ce notebook est le **coeur solveur** du capstone MealPlanner : il porte **fidelement**
la methode `PlanificateurDeMenus.Create` du Demo2 d'origine
([fork Z3.Linq, branche `EPFdevelopment`](https://github.com/MyIntelligenceAgency/Z3.LinqBinding)).

La **Slice A** ([05c](05c_Dietetique_Load_Faithful_Port.ipynb)) a construit le corpus derive
(`dietetique.json` : fusion Ciqual ANSES x Recettes via `Dietetique.Load`). **Aucun raisonnement
SMT** ne s'y produisait — c'etait de la **preparation de donnees**. Ici, Z3 entre en scene : on
**resout** un plan de menus qui satisfait les **restrictions nutritionnelles d'un patient**.

## La hierarchie a 4 niveaux (fidele a la source)

```
Patient (restrictions Min/Max par constituant)
   |
   +-- Menu 1 .. Menu N           (un menu = un jour)
          |
          +-- Plat 1 .. Plat P    (un plat par creneau : ordre 1, 2, 3, ...)
                 |
                 +-- Denrees      (composition nutritionnelle reelle, vecteur Ciqual)
```

La nutrition d'un menu est la **somme** des compositions de ses plats ; chaque plat apporte le
vecteur de teneurs (74 constituants) calcule en Slice A. Le **patient** impose des bornes
`Min`/`Max` par constituant (ex : energie entre 800 et 2600 kcal, lipides plafonnees) — exactement
la classe `Restriction { Min, Max }` du source.

## Ce que ce notebook ajoute a 05/05b

- [05](05_Meal_Planner_Hierarchical.ipynb) / [05b](05b_Meal_Planner_Real_Corpus_Hierarchical.ipynb)
  posent le squelette `int[][]` (bornes de creneau + `Distinct` = variete).
- **05d ajoute la couche nutritionnelle FIDELE** absente jusqu'ici : le **linking de composition**
  (`PlatId != candidat || Comp == teneur(candidat)`, source lignes 60-99) **et** les **restrictions
  patient Min/Max** sur la somme par menu. C'est le chainon qui relie le corpus reel (05c) au
  raisonnement Z3.


## 0. Dependances et corpus

Le notebook charge le cache `dietetique.json` produit par la **Slice A** ([05c](05c_Dietetique_Load_Faithful_Port.ipynb)).
Si le cache est absent : executer 05c (ou `python download_meal_data.py` puis 05c). **Echec bruyant**, pas
de corpus de substitution (regle SOTA : on raisonne sur les vraies donnees nutritionnelles).

Le fork Z3.Linq est charge depuis `.deploy/` (script de build unique, cf README) — meme build que
toute la serie, incluant le support `int[][]` et `CollectionHandling.Array`.

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Text.Json;
using System.Diagnostics;

// POCOs miroir du schema dietetique.json (couche de fusion Ciqual x Recettes, Slice A).
public class Dietetique {
    public List<Constituant> Constituants { get; set; }
    public List<DenreeImport> Denrees { get; set; }
    public List<PlatImport> Plats { get; set; }
}
public class Constituant { public string Nom { get; set; } }
public class DenreeImport { public string Nom { get; set; } = ""; public decimal[] Compositions { get; set; } }
public class Ingredient { public int Denree { get; set; } public decimal Quantite { get; set; } }
public class PlatImport { public string Nom { get; set; } public int Ordre { get; set; } public decimal[] Compositions { get; set; } }

// Resolution robuste du cache (relatif au notebook, repli remontant) ; affichage relatif (pas de chemin machine).
string ResolveDiet() {
    foreach (var c in new[]{ "data/meals/dietetique.json",
                             "MyIA.AI.Notebooks/SymbolicAI/SMT/Z3/data/meals/dietetique.json" })
        if (File.Exists(c)) return c;
    return "data/meals/dietetique.json";
}
var dietPath = ResolveDiet();
Dietetique diet = null;
if (!File.Exists(dietPath)) {
    Console.WriteLine("[CACHE ABSENT] " + dietPath);
    Console.WriteLine("Executez d'abord 05c (Slice A) qui produit dietetique.json,");
    Console.WriteLine("ou : python download_meal_data.py  puis 05c.");
} else {
    diet = JsonSerializer.Deserialize<Dietetique>(File.ReadAllText(dietPath));
    Console.WriteLine($"Corpus charge : {diet.Constituants.Count} constituants / "
                      + $"{diet.Denrees.Count} denrees / {diet.Plats.Count} plats");
}


The below script needs to be able to find the current output cell; this is an easy method to get it.

Corpus charge : 74 constituants / 198 denrees / 138 plats


## 1. Curation d'un sous-ensemble tractable

La source pose le theoreme sur **138 plats x 74 constituants x 7 menus x 5 creneaux**. Le linking de
composition genere alors `7 x 5 x 138 x 74 ~ 357 000` assertions — l'instance **explose** (c'est
precisement le probleme de *convergence a l'echelle*, objet de la **Slice D**). Le corps de l'issue
#4617 le prevoit : **on cure d'abord un sous-ensemble**.

On reduit donc a **2 menus x 3 creneaux**, **5 candidats par creneau**, et **3 constituants cles**
(energie kcal, proteines, lipides). Le theoreme reste **fidele dans sa structure** ; seules les
*dimensions* sont reduites pour rester pedagogiquement lisible et resoluble en quelques secondes.

In [2]:
// --- Parametres du sous-ensemble cure ---
const int NbMenus = 2;          // 2 jours
const int NbPlats = 3;          // 3 creneaux (ordre 1, 2, 3)
const int CandParOrdre = 5;     // 5 candidats par creneau

// 3 constituants cles, reperes par NOM (robuste a l'ordre du fichier)
int cEnergie = diet.Constituants.FindIndex(c => c.Nom.Contains("kcal/100 g") && c.Nom.Contains("1169"));
int cProt    = diet.Constituants.FindIndex(c => c.Nom.ToLowerInvariant().Contains("prot"));
int cLip     = diet.Constituants.FindIndex(c => c.Nom.ToLowerInvariant().Contains("lipides"));
int[] CONST  = new[]{ cEnergie, cProt, cLip };
string[] NOMK = new[]{ "energie(kcal)", "proteines(g)", "lipides(g)" };
Console.WriteLine("Constituants suivis :");
foreach (var ci in CONST) Console.WriteLine($"  [{ci}] {diet.Constituants[ci].Nom.Trim()}");

// Pool curee : CandParOrdre plats par ordre 1..NbPlats, re-indexes 0..(NbPlats*CandParOrdre-1).
// Le creneau p (0-base) ne propose que les plats d'ordre (p+1) -> indices [p*CandParOrdre, (p+1)*CandParOrdre).
var pool = new List<PlatImport>();
for (int o = 1; o <= NbPlats; o++)
    pool.AddRange(diet.Plats.Where(p => p.Ordre == o).Take(CandParOrdre));
Console.WriteLine($"\nPool curee : {pool.Count} plats ({NbPlats} creneaux x {CandParOrdre} candidats)");

// teneur arrondie a l'entier (arithmetique entiere Z3) ; helpers d'indexation aplatie
int Teneur(int poolIdx, int constIdx) => (int)Math.Round(pool[poolIdx].Compositions[constIdx]);
int Slot(int m, int p) => m * NbPlats + p;   // ligne aplatie pour le slot (menu m, creneau p)
Console.WriteLine("\nApercu (creneau : candidats -> energie kcal) :");
for (int p = 0; p < NbPlats; p++) {
    var noms = Enumerable.Range(p*CandParOrdre, CandParOrdre)
        .Select(i => $"{pool[i].Nom.Trim()}({Teneur(i, cEnergie)})");
    Console.WriteLine($"  creneau {p+1} : " + string.Join(", ", noms));
}


Constituants suivis :


  [1] Energie, Règlement UE N° 1169/2011 (kcal/100 g)


  [19] Protéines, N x facteur de Jones (g/100 g)


  [32] Lipides (g/100 g)



Pool curee : 15 plats (3 creneaux x 5 candidats)



Apercu (creneau : candidats -> energie kcal) :


  creneau 1 : RADIS BEURRE(11), SAL MELANGE TENDRE(79), PAIN BAGNAT ECOLE(253), GOUGERE INDIVIDUELLE(0), SALADE KOUKI AU CURRY(1366)


  creneau 2 : DAUBE BBC(1889), SAUCISSE VEGETALE(398), ROTI DE VEAU (cuisson)(491), POISSON PANE CUIT / CITRON(275), NUGGETS VEGETAL(247)


  creneau 3 : RIZ CAMARGUAIS(1455), TORSADES HALLOWEEN(2026), P DE TERRE VAPEUR(1033), CAROTTE RONDELLES (BTE)(1310), GRATIN DE COURGETTES A LA TOMATE(1354)


## 2. Le modele : 3 tableaux `int[][]`

Le Demo2 d'origine modelise un **graphe d'objets** (`Menu[]` contenant `Plat[]` contenant `PlatId` +
`Compositions[]`). Le binding Z3.Linq de la serie expose, lui, des **tableaux `int[][]`** (cf 05/05b) :
c'est l'encodage *index* que le fork resout en notebook. On reproduit donc la **semantique** de la
source avec trois `int[][]` :

| Tableau | Forme | Role |
|---------|-------|------|
| `PlatId[m][p]`  | `[NbMenus][NbPlats]`          | index (dans la pool) du plat choisi au creneau (m, p) |
| `Comp[slot][k]` | `[NbMenus*NbPlats][3]`        | composition du plat choisi a ce slot (variable liee par disjonction) |
| `MenuNut[m][k]` | `[NbMenus][3]`                | somme nutritionnelle du menu m (ce que le patient contraint) |

`Comp` est l'**aux variable** qui resout l'impossibilite d'indexer un tableau C# par une variable Z3
(on ne peut pas ecrire `Plats[PlatId[m][p]]`) : on la **lie** au bon plat par une disjonction
(section 3.3), exactement la technique de la source.

In [3]:
public class PlanRepas {
    public int[][] PlatId  { get; set; }   // [NbMenus][NbPlats]      : plat choisi par creneau
    public int[][] Comp    { get; set; }   // [NbMenus*NbPlats][3]    : composition liee du slot
    public int[][] MenuNut { get; set; }   // [NbMenus][3]            : somme par menu
    public PlanRepas() {
        PlatId  = Enumerable.Range(0, 2).Select(_ => new int[3]).ToArray();
        Comp    = Enumerable.Range(0, 6).Select(_ => new int[3]).ToArray();
        MenuNut = Enumerable.Range(0, 2).Select(_ => new int[3]).ToArray();
    }
}
Console.WriteLine("Modele PlanRepas defini (PlatId, Comp, MenuNut).");


Modele PlanRepas defini (PlatId, Comp, MenuNut).


## 3. Le patient et ses restrictions

Fidele a la classe `Patient { Restriction[] Restrictions }` du source : chaque restriction porte un
`Min` et un `Max` par constituant (`-1` = pas de borne). Ici, le patient impose une **fenetre
energetique** par menu et un **plafond de lipides**.

In [4]:
// Restrictions patient (fidele a Restriction { Min, Max }, -1 = pas de borne), par MENU.
// Echelle : compositions Ciqual en /100 g sommees sur les 3 plats du menu.
int energieMin = 800;    // kcal cumules minimum par menu
int energieMax = 2600;   // kcal cumules maximum par menu
int protMin    = 30;     // proteines minimum par menu
int lipMax     = 600;    // lipides maximum par menu
Console.WriteLine($"Patient : energie in [{energieMin}, {energieMax}] kcal, proteines >= {protMin} g, lipides <= {lipMax} g (par menu)");


Patient : energie in [800, 2600] kcal, proteines >= 30 g, lipides <= 600 g (par menu)


### 3.1 -> 3.5 Construction du theoreme (port fidele de `Create`)

Les cinq familles de contraintes reproduisent `PlanificateurDeMenus.Create` (source lignes 38-101) :

1. **Bornes = ordre** : `PlatId[m][p]` indexe un plat du creneau (p+1) — la source impose
   `Dietetique.Plats[rid].Ordre == p+1` ; ici la partition de la pool par ordre l'encode dans les bornes.
2. **Variete** : `Z3Methods.Distinct(...)` sur tous les slots — *pas deux fois le meme plat*
   (source : `Distinct` sur la grille des `PlatId`).
3. **Linking composition** : `PlatId[m][p] != cand || Comp[slot][k] == teneur(cand, k)` — lie la
   variable de composition au plat reellement choisi (source lignes 71-74).
4. **Somme par menu** : `MenuNut[m][k] == somme_p Comp[slot(m,p)][k]` (source lignes 84-85).
5. **Restrictions patient** : `Min < MenuNut[m][k] < Max` selon les bornes du patient (source lignes 90-99).

In [5]:
var sw = Stopwatch.StartNew();
PlanRepas sol = null;
using (var ctx = new Z3Context { DefaultCollectionHandling = CollectionHandling.Array })
{
    var th = ctx.NewTheorem<PlanRepas>();

    // (1) bornes = contrainte d'ordre : slot p -> candidats du creneau (p+1)
    for (int m = 0; m < NbMenus; m++) for (int p = 0; p < NbPlats; p++) {
        int mm = m, pp = p, lo = p * CandParOrdre, hi = p * CandParOrdre + CandParOrdre - 1;
        th = th.Where(x => x.PlatId[mm][pp] >= lo && x.PlatId[mm][pp] <= hi);
    }

    // (2) variete : Distinct global sur les 6 slots
    th = th.Where(x => Z3Methods.Distinct(
        x.PlatId[0][0], x.PlatId[0][1], x.PlatId[0][2],
        x.PlatId[1][0], x.PlatId[1][1], x.PlatId[1][2]));

    // (3) linking composition : PlatId[m][p] != cand || Comp[slot][k] == teneur(cand, k)
    for (int m = 0; m < NbMenus; m++) for (int p = 0; p < NbPlats; p++) {
        int slot = Slot(m, p), mm = m, pp = p;
        for (int cand = p * CandParOrdre; cand < p * CandParOrdre + CandParOrdre; cand++) {
            int cc = cand;
            for (int k = 0; k < CONST.Length; k++) {
                int kk = k, val = Teneur(cand, CONST[k]);
                th = th.Where(x => x.PlatId[mm][pp] != cc || x.Comp[slot][kk] == val);
            }
        }
    }

    // (4) somme par menu : MenuNut[m][k] == Comp[s0][k] + Comp[s1][k] + Comp[s2][k]
    for (int m = 0; m < NbMenus; m++) for (int k = 0; k < CONST.Length; k++) {
        int mm = m, kk = k, s0 = Slot(m, 0), s1 = Slot(m, 1), s2 = Slot(m, 2);
        th = th.Where(x => x.MenuNut[mm][kk] == x.Comp[s0][kk] + x.Comp[s1][kk] + x.Comp[s2][kk]);
    }

    // (5) restrictions patient : energie in [min,max], proteines >= min, lipides <= max (par menu)
    for (int m = 0; m < NbMenus; m++) {
        int mm = m;
        th = th.Where(x => x.MenuNut[mm][0] >= energieMin && x.MenuNut[mm][0] <= energieMax);
        th = th.Where(x => x.MenuNut[mm][1] >= protMin);
        th = th.Where(x => x.MenuNut[mm][2] <= lipMax);
    }

    sol = th.Solve();
}
sw.Stop();
Console.WriteLine($"Theoreme resolu en {sw.Elapsed.TotalMilliseconds:F0} ms\n");
if (sol == null) {
    Console.WriteLine("UNSAT : aucun plan ne satisfait les restrictions du patient sur cette pool.");
} else {
    for (int m = 0; m < NbMenus; m++) {
        Console.WriteLine($"== Menu {m + 1} ==  energie={sol.MenuNut[m][0]} kcal | "
            + $"proteines={sol.MenuNut[m][1]} g | lipides={sol.MenuNut[m][2]} g");
        for (int p = 0; p < NbPlats; p++) {
            var plat = pool[sol.PlatId[m][p]];
            Console.WriteLine($"   creneau {p + 1} : {plat.Nom.Trim(),-30} "
                + $"(kcal={Teneur(sol.PlatId[m][p], cEnergie)}, prot={Teneur(sol.PlatId[m][p], cProt)}, lip={Teneur(sol.PlatId[m][p], cLip)})");
        }
    }
}


Theoreme resolu en 120 ms



== Menu 1 ==  energie=2596 kcal | proteines=69 g | lipides=164 g


   creneau 1 : SAL MELANGE TENDRE             (kcal=79, prot=1, lip=0)


   creneau 2 : ROTI DE VEAU (cuisson)         (kcal=491, prot=37, lip=13)


   creneau 3 : TORSADES HALLOWEEN             (kcal=2026, prot=31, lip=151)


== Menu 2 ==  energie=1864 kcal | proteines=40 g | lipides=128 g


   creneau 1 : RADIS BEURRE                   (kcal=11, prot=1, lip=0)


   creneau 2 : SAUCISSE VEGETALE              (kcal=398, prot=16, lip=37)


   creneau 3 : RIZ CAMARGUAIS                 (kcal=1455, prot=23, lip=91)


## 4. Verification : les restrictions sont-elles vraiment respectees ?

Un solveur **garantit** les contraintes par construction — mais on **verifie** independamment
(re-calcul hors Z3) que chaque menu respecte la fenetre du patient. C'est la difference entre *croire*
le solveur et *prouver* le resultat.

In [6]:
if (sol == null) {
    Console.WriteLine("Pas de solution a verifier (UNSAT).");
} else {
    bool ok = true;
    for (int m = 0; m < NbMenus; m++) {
        // re-somme directe depuis la pool (independante des variables Z3)
        int e = 0, pr = 0, li = 0;
        for (int p = 0; p < NbPlats; p++) {
            e  += Teneur(sol.PlatId[m][p], cEnergie);
            pr += Teneur(sol.PlatId[m][p], cProt);
            li += Teneur(sol.PlatId[m][p], cLip);
        }
        bool eOk = e >= energieMin && e <= energieMax, pOk = pr >= protMin, lOk = li <= lipMax;
        bool match = e == sol.MenuNut[m][0] && pr == sol.MenuNut[m][1] && li == sol.MenuNut[m][2];
        ok &= eOk && pOk && lOk && match;
        Console.WriteLine($"Menu {m+1} : recalc energie={e}({(eOk?"OK":"HORS")}), prot={pr}({(pOk?"OK":"HORS")}), "
            + $"lip={li}({(lOk?"OK":"HORS")}) ; coherent avec MenuNut={(match?"oui":"NON")}");
    }
    Console.WriteLine(ok ? "\nVERIFIE : toutes les restrictions patient sont respectees."
                         : "\nINCOHERENCE detectee (a investiguer).");
}


Menu 1 : recalc energie=2596(OK), prot=69(OK), lip=164(OK) ; coherent avec MenuNut=oui


Menu 2 : recalc energie=1864(OK), prot=40(OK), lip=128(OK) ; coherent avec MenuNut=oui



VERIFIE : toutes les restrictions patient sont respectees.


## 5. Exercices

Trois exercices pour s'approprier le theoreme hierarchique a restrictions patient. Les stubs
s'executent tels quels (ils n'echouent pas) ; completez le corps marque `TODO`.

### Exercice 1 — Une restriction qui rend le probleme insatisfiable

Le patient devient diabetique : imposez un **plafond energetique strict** par menu (ex. `energieMax = 1500`)
et reconstruisez le theoreme. Observez si Z3 trouve encore un plan ou repond **UNSAT**, et expliquez
*pourquoi* (quel creneau porte l'essentiel des kcal ?).

*Indice* : recopiez le bloc de la section 3.5 en changeant `energieMax`, ou parametrez-le.

In [7]:
// Exercice 1 : trouver le plafond energetique qui bascule SAT -> UNSAT.
// TODO etudiant : reconstruire le theoreme avec un energieMax plus strict et reporter le verdict.
int ExerciceUnsatThreshold()
{
    // Etape 1 : choisir un energieMax candidat (ex. 1500)
    // Etape 2 : reconstruire le theoreme (sections 1-5 bornes/Distinct/linking/somme/restrictions)
    //           en remplacant la borne energieMax
    // Etape 3 : retourner le plus grand energieMax pour lequel le probleme est UNSAT (ou -1 si toujours SAT)
    Console.WriteLine("Exercice 1 a completer");
    return -1;
}
ExerciceUnsatThreshold();


Exercice 1 a completer


### Exercice 2 — Passer a 3 menus (tractabilite)

Portez `NbMenus` de 2 a **3** (3 jours). Adaptez la taille des tableaux du modele `PlanRepas`
(`PlatId` `[3][3]`, `Comp` `[9][3]`, `MenuNut` `[3][3]`) et le `Distinct` (9 slots). Mesurez le temps
de resolution : reste-t-il de l'ordre de la seconde ? C'est un premier pas vers la **Slice D**
(convergence a l'echelle).

*Indice* : `Z3Methods.Distinct` accepte une liste d'arguments ; enumerez les 9 `PlatId[m][p]`.

In [8]:
// Exercice 2 : etendre a NbMenus = 3 et mesurer le temps de resolution.
// TODO etudiant : adapter le modele (tailles), le Distinct (9 slots) et resoudre.
double ExerciceTroisMenus()
{
    // Etape 1 : definir un modele PlanRepas3 avec PlatId[3][3], Comp[9][3], MenuNut[3][3]
    // Etape 2 : reconstruire les 5 familles de contraintes pour 3 menus
    // Etape 3 : chronometrer Solve() et retourner le temps en millisecondes (ou -1 si UNSAT)
    Console.WriteLine("Exercice 2 a completer");
    return -1.0;
}
ExerciceTroisMenus();


Exercice 2 a completer


### Exercice 3 — Ajouter un quatrieme constituant suivi

Le patient surveille aussi le **sel** (`Sel chlorure de sodium`). Ajoutez ce constituant a `CONST`
(repere par nom), etendez `Comp`/`MenuNut` a 4 colonnes, et imposez un **plafond de sel** par menu.
Le plan reste-t-il satisfiable ?

*Indice* : `diet.Constituants.FindIndex(c => c.Nom.ToLowerInvariant().Contains("sel chlorure"))`.

In [9]:
// Exercice 3 : suivre un 4e constituant (sel) et lui imposer un plafond par menu.
// TODO etudiant : etendre CONST a 4 elements et ajouter la restriction de sel.
int ExerciceConstituantSel()
{
    // Etape 1 : reperer l'index du constituant "Sel chlorure de sodium"
    // Etape 2 : etendre CONST/Comp/MenuNut a 4 colonnes
    // Etape 3 : ajouter Where(MenuNut[m][3] <= selMax) et retourner le sel max du menu 1 de la solution
    Console.WriteLine("Exercice 3 a completer");
    return 0;
}
ExerciceConstituantSel();


Exercice 3 a completer


## Conclusion

Ce notebook a porte **fidelement** le coeur solveur du Demo2 (`PlanificateurDeMenus.Create`) : un
**theoreme hierarchique a 4 niveaux** (Patient -> Menus -> Plats -> Denrees), avec **linking de
composition** (la variable nutritionnelle liee au plat reellement choisi) et **restrictions patient
Min/Max** sur la somme par menu — le tout sur le **corpus reel** Ciqual x Recettes construit en
[Slice A](05c_Dietetique_Load_Faithful_Port.ipynb).

La place de ce notebook dans le capstone #4617 :

| Slice | Notebook | Apport |
|-------|----------|--------|
| A | [05c](05c_Dietetique_Load_Faithful_Port.ipynb) | construction du corpus (`Dietetique.Load`, 0 SMT) |
| B | [07b](07b_Meal_Planner_Real_Corpus_Scaling.ipynb) | RecipeML a l'echelle (3 encodages SMT) |
| **C** | **05d (ce notebook)** | **theoreme hierarchique + restrictions patient (port fidele)** |
| D | *a venir* | convergence a l'echelle (138 plats x 74 constituants) |

**Vers la Slice D** : ici on a **cure** un sous-ensemble (2 menus, 3 creneaux, 5 candidats, 3
constituants) pour rester resoluble en quelques secondes. Le passage au corpus complet fait exploser
le **linking** (`~357 000` assertions) — c'est le probleme de *convergence a l'echelle* que la Slice D
adressera (encodages alternatifs, pseudo-booleens a la [07b](07b_Meal_Planner_Real_Corpus_Scaling.ipynb),
pre-filtrage des candidats).
